# Creating the Chunker for the Documents

#### The Documents that would be covered natively are 

- Markdown
- txt
- docx
- xlxs / csv
- pdf
- pptx
- json / yaml

#### Other files that would be created would have the 300 text + 20 overlap rule, and each of the supported file would also have the simillar chunk theory 300 word and 20 overlap

In [2]:
import re
import json
from pygments.lexers import get_lexer_for_filename
from pygments.util import ClassNotFound
from pathlib import Path
from tree_sitter_language_pack import get_parser, get_language
from tree_sitter import Query, Node, QueryCursor, Tree

In [ ]:

def extract_structure(file_path):
    structure = []
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line_num, line in enumerate(file, 1):
            # Heading pattern match kar rahe hain (# Heading)
            match = re.match(r'^(#{1,6})\s+(.*)', line.strip())
            if match:
                level = len(match.group(1))  # Kitne '#' hain (1 to 6)
                title = match.group(2)
                
                # Indentation ke sath structure format kar rahe hain
                indent = "  " * (level - 1)
                structure.append(f"{indent}- H{level}: {title} (Line {line_num})")
                
    return "\n".join(structure)

# Usage
file_name = "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md"  # Aapki file ka path
outline = extract_structure(file_name)
print("=== File Content Structure ===")
print(outline)

In [4]:
def parse_markdown_sections(markdown_text: str):
    lines = markdown_text.split("\n")

    headers = {}
    content = []
    in_code_block = False
    parsed_file = []

    for line in lines:
        stripped_line = line.strip()

        if stripped_line.startswith("```") or stripped_line.startswith("~~~"):
            in_code_block = not in_code_block
            content.append(line)
            continue

        match_header = re.match(r"^(#{1,6})\s+(.*)", stripped_line) if not in_code_block else None
        if match_header:

            if content:
                parsed_file.append({"headers": headers, "content": '\n'.join(content).strip()})
                content = []

            level = len(match_header.group(1))
            title = match_header.group(2).strip()

            headers = {k: v for k, v in headers.items() if k < level}
            headers[level] = title
        else:
            content.append(stripped_line)

    if content:
        parsed_file.append({"headers": headers, "content": '\n'.join(content).strip()})

    return parsed_file

In [5]:
file_path = (
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.md"
)

file = None

with open(file_path, "r", encoding="utf-8") as f:
    file = f.read()


content = parse_markdown_sections(file)
# for c in content:
#     print(c)


In [6]:
def extension_to_language_name(file_name: Path, get_full_name: bool = False) -> str:
    try:
        lexer = get_lexer_for_filename(file_name)
        if get_full_name:
            return lexer.name
        return lexer.aliases[0] if lexer.aliases else "text"
    except ClassNotFound:
        return "unknown"  # Fallback agar extension parse na ho paaye

In [7]:
def parse_ast(file_path: Path):
    file_alias = extension_to_language_name(file_path)
    if not file_alias:
        return None

    lang_name = file_alias
    if not lang_name:
        return None

    try:
        parser = get_parser(lang_name)
        language = get_language(lang_name)
        print(file_alias)
        print(language)


        source_code = file_path.read_text(encoding="utf-8")
        tree = parser.parse(bytes(source_code, "utf-8"))

        return tree
    except Exception:
        print(f"Skipping {file_path.name}: No parser found for language '{lang_name}'")
        return None

In [8]:
def print_ast(node, source_bytes: bytes, indent: str = "", is_last: bool = True):
    if node is None:
        return

    marker = "└── " if is_last else "├── "

    # Extract source code string for leaf nodes (nodes with no children)
    snippet = ""
    if len(node.children) == 0:
        token_text = source_bytes[node.start_byte : node.end_byte].decode(
            "utf-8", errors="replace"
        )
        snippet = f" ➔ {token_text!r}"

    # Format line and column coordinates
    pos = f"[{node.start_point[0] + 1}:{node.start_point[1]}]"

    print(f"{indent}{marker}{node.type} {pos}{snippet}")

    # Prepare indentation string for children
    new_indent = indent + ("    " if is_last else "│   ")

    # Recursively traverse child nodes
    count = len(node.children)
    for i, child in enumerate(node.children):
        print_ast(child, source_bytes, new_indent, is_last=(i == count - 1))

In [9]:
tree = parse_ast(Path(file_path))
if tree:
    source_byte = Path(file_path).read_bytes()
    # print_ast(tree.root_node, source_byte)

markdown
<Language id=129931015998688, version=14, name=None>


In [10]:
def _text(node, source_bytes):
    return source_bytes[node.start_byte:node.end_byte].decode("utf-8")


In [11]:
def parse_markdown(parent_node: Node, metadata=None):
    if metadata is None:
        metadata = {}

    chunks = []

    for node in parent_node.children:
        print(node)
        if node.type == "section":
            current_metadata = metadata.copy()

            current_metadata["headings"] = current_metadata.get("headings", {}).copy()

            is_image = False
            is_code_block = False
            is_table = False
            for child in node.children:

                if child.type == "atx_heading":
                    content = _text(child, source_byte)

                    level = len(content) - len(content.lstrip("#"))

                    current_metadata["headings"] = {
                        k: v
                        for k, v in current_metadata["headings"].items()
                        if k < level
                    }

                    current_metadata["headings"][level] = content

                elif child.type == "paragraph":
                    paragraph_content = _text(child, source_byte)

                    match_para = re.match(
                        r'!\["?(.*?)"?\]\("?(.*?)"?\)', paragraph_content
                    )
                    if match_para:
                        is_image = True

                elif child.type == "html_block":
                    html = _text(child, source_byte)
                    match_html = re.search(
                        r'<img\b[^>]*\bsrc=["\']([^"\']*)["\'][^>]*\balt=["\']([^"\']*)["\']',
                        html,
                        re.IGNORECASE,
                    )
                    if match_html:
                        is_image = True
                    # print("HTML Block:", html)

                elif child.type == "fenced_code_block":
                    is_code_block = True

                elif child.type == "pipe_table":
                    is_table = True
                    # print(_text(child, source_byte))

            chunks.append(
                {
                    "metadata": {
                        **current_metadata,
                        "is_image": is_image,
                        "is_code_block": is_code_block,
                        "is_table": is_table,
                    }
                }
            )

            chunks.extend(parse_markdown(node, current_metadata))

        else:
            # chunks.extend(parse_markdown(node, metadata))
            pass

    return chunks

In [12]:
def vission_LLM(image_url):
    """
    Placeholder for your Vision LLM function.
    It should take an image URL and return a text description.
    """
    # Replace this with your actual LLM calling logic
    return f"[Image Description: A detailed description of the image at {image_url}]"


def parse_markdown(parent_node, metadata=None):
    if metadata is None:
        metadata = {}

    chunks = []

    for node in parent_node.children:
        # print(node) # Uncomment for debugging
        if node.type == "section":
            current_metadata = metadata.copy()
            current_metadata["headings"] = current_metadata.get("headings", {}).copy()

            is_image = False
            is_code_block = False
            is_table = False

            # Arrays to hold content parts and image metadata for the current section
            section_content = []
            image_urls = []
            alt_texts = []

            for child in node.children:
                # print("Child Type, " , child.type, "Text, ", child.text)
                if child.type == "atx_heading":
                    content = _text(child, source_byte)
                    level = len(content) - len(content.lstrip("#"))

                    current_metadata["headings"] = {
                        k: v
                        for k, v in current_metadata["headings"].items()
                        if k < level
                    }
                    current_metadata["headings"][level] = content

                    # Add heading text to the content
                    section_content.append(content)

                elif child.type == "paragraph":
                    paragraph_content = _text(child, source_byte)

                    # Use regex to find all markdown images: ![alt](url)
                    img_pattern = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")

                    if img_pattern.search(paragraph_content):
                        is_image = True

                        def replace_md_image(match):
                            alt = match.group(1)
                            url = match.group(2)
                            image_urls.append(url)
                            alt_texts.append(alt)

                            # Call the Vision LLM to get the description
                            description = vission_LLM(url)
                            return description

                        # Replace the image markdown with the LLM description
                        paragraph_content = img_pattern.sub(
                            replace_md_image, paragraph_content
                        )

                    html_img_pattern = re.compile(
                        r'<img\b(?=[^>]*\bsrc\s*=\s*["\']([^"\']*)["\'])'
                        r'(?=[^>]*\balt\s*=\s*["\']([^"\']*)["\'])'
                        r"[^>]*>",
                        re.IGNORECASE | re.DOTALL,
                    )

                    if html_img_pattern.search(paragraph_content):
                        print("HTML image found in paragraph:", paragraph_content)
                        is_image = True

                        def replace_html_image(match):
                            url = match.group(1)  # src
                            alt = match.group(2)  # alt
                            image_urls.append(url)
                            alt_texts.append(alt)

                            # Call the Vision LLM to get the description
                            description = vission_LLM(url)
                            return description

                        # Replace the HTML img tag with the LLM description
                        paragraph_content = html_img_pattern.sub(
                            replace_html_image, paragraph_content
                        )

                    # section_content.append(html)

                    section_content.append(paragraph_content)

                elif child.type == "html_block":
                    print("Enter the html_block", child.text)
                    html = _text(child, source_byte)

                    # Use regex to find all HTML images: <img src="url" alt="alt">
                    html_img_pattern = re.compile(
                        r'<img\b(?=[^>]*\bsrc\s*=\s*["\']([^"\']*)["\'])'
                        r'(?=[^>]*\balt\s*=\s*["\']([^"\']*)["\'])'
                        r"[^>]*>",
                        re.IGNORECASE | re.DOTALL,
                    )

                    if html_img_pattern.search(html):
                        is_image = True

                        def replace_html_image(match):
                            url = match.group(1)  # src
                            alt = match.group(2)  # alt
                            image_urls.append(url)
                            alt_texts.append(alt)

                            # Call the Vision LLM to get the description
                            description = vission_LLM(url)
                            return description

                        # Replace the HTML img tag with the LLM description
                        html = html_img_pattern.sub(replace_html_image, html)

                    section_content.append(html)

                elif child.type == "fenced_code_block":
                    is_code_block = True
                    # Add code block directly to content
                    section_content.append(_text(child, source_byte))

                elif child.type == "pipe_table":
                    is_table = True
                    # Add the raw markdown table directly to the content
                    section_content.append(_text(child, source_byte))

                elif child.type == "section":
                    # Skip nested sections in this loop.
                    # They will be processed by the recursive call below.
                    pass

                else:
                    # Catch-all for lists, block_quotes, thematic_breaks, etc.
                    # Adds their text directly to the content.
                    section_content.append(_text(child, source_byte))

            # Compile metadata for this section chunk
            chunk_metadata = {
                **current_metadata,
                "is_image": is_image,
                "is_code_block": is_code_block,
                "is_table": is_table,
            }

            # If images were found, add their URLs and alts to the metadata
            if is_image:
                chunk_metadata["image"] = {
                    "image_urls": image_urls,
                    "alt_texts": alt_texts,
                }

            # Join all collected content pieces with double newlines for readability
            final_content = "\n\n".join(section_content).strip()

            # Append the completed chunk
            if final_content:
                chunks.append({"metadata": chunk_metadata, "content": final_content})

            # Recursively process any nested sections
            chunks.extend(parse_markdown(node, current_metadata))

        else:
            # Handle root-level nodes that aren't inside a section (if any exist)
            parse_markdown(node, metadata)

    return chunks

In [ ]:
if tree:
    store = parse_markdown(tree.root_node)


In [ ]:
print(store)

In [15]:

def count_words(text: str) -> int:
    return len(re.findall(r'\w+', text))



def split_image_chunk(chunk: dict) -> list:
    """Splits a single chunk containing multiple images into localized sliding windows."""
    meta = chunk.get("metadata", {})
    
    # Handle both nested "image" dict and flat structures dynamically
    if "image" in meta and isinstance(meta["image"], dict):
        urls = meta["image"].get("image_urls", [])
        alts = meta["image"].get("alt_texts", [])
        is_nested = True
    else:
        urls = meta.get("image_urls", [])
        alts = meta.get("alt_texts", [])
        is_nested = False
        
    content = chunk.get("content", "")
    
    if len(urls) <= 1:
        chunk["word_count"] = count_words(content) 
        return [chunk]
        
    split_chunks = []
    lines = content.split('\n')
    
    # 1. Map each URL to its exact line number in the content
    url_indices = []
    for url in urls:
        for idx, line in enumerate(lines):
            if url in line:
                url_indices.append((url, idx))
                break 

    # 2. Create a localized sliding window for each image
    for i, (target_url, target_idx) in enumerate(url_indices):
        new_meta = meta.copy()
        
        if is_nested:
            new_meta["image"] = {
                "image_urls": [target_url],
                "alt_texts": [alts[i]] if i < len(alts) else []
            }
        else:
            new_meta["image_urls"] = [target_url]
            new_meta["alt_texts"] = [alts[i]] if i < len(alts) else []
            
        # --- LOCALIZED WINDOW LOGIC ---
        # Start just after the PREVIOUS image (or at the very beginning)
        start_idx = url_indices[i-1][1] + 1 if i > 0 else 0
        
        # End right at the NEXT image (or at the very end)
        end_idx = url_indices[i+1][1] if i < len(url_indices) - 1 else len(lines)
        
        # Slice the lines for this specific image
        chunk_lines = lines[start_idx:end_idx]
        
        new_content = "\n".join(chunk_lines).strip()
        new_content = re.sub(r'\n{3,}', '\n\n', new_content)
        
        split_chunks.append({
            "metadata": new_meta,
            "content": new_content,
            "word_count": count_words(new_content)
        })
        
    return split_chunks
def split_table_chunk(chunk: dict, max_table_rows: int) -> list:
    """Splits large markdown tables into smaller tables, repeating the headers."""
    meta = chunk.get("metadata", {})
    content = chunk.get("content", "")
    
    lines = content.split('\n')
    table_lines, before_table, after_table = [], [], []
    
    in_table = False
    for line in lines:
        if '|' in line and line.strip().startswith('|'):
            in_table = True
            table_lines.append(line)
        else:
            if in_table:
                after_table.append(line)
            else:
                before_table.append(line)
                
    if len(table_lines) <= max_table_rows + 2 or not table_lines:
        chunk["word_count"] = count_words(content)
        return [chunk]
        
    header_lines = table_lines[:2]
    body_lines = table_lines[2:]
    
    split_chunks = []
    for i in range(0, len(body_lines), max_table_rows):
        chunk_body = body_lines[i:i+max_table_rows]
        
        new_lines = list(before_table)
        new_lines.extend(header_lines)
        new_lines.extend(chunk_body)
        
        if i + max_table_rows >= len(body_lines):
            new_lines.extend(after_table)
            
        new_content = "\n".join(new_lines).strip()
        split_chunks.append({
            "metadata": meta.copy(),
            "content": new_content,
            "word_count": count_words(new_content)
        })
        
    return split_chunks

def flush_text_buffer(merged_chunks, current_buffer, overlap_chunks):
    if not current_buffer:
        return

    # Combine the text
    merged_content = "\n\n".join(c["content"] for c in current_buffer)
    total_words = sum(c["word_count"] for c in current_buffer)

    # Smart Metadata Aggregation: 
    # Collect all unique H1s and H2s touched in this merged chunk so context isn't lost
    all_h1s = list(dict.fromkeys(c.get("metadata", {}).get("headings", {}).get(1) for c in current_buffer if c.get("metadata", {}).get("headings", {}).get(1)))
    all_h2s = list(dict.fromkeys(c.get("metadata", {}).get("headings", {}).get(2) for c in current_buffer if c.get("metadata", {}).get("headings", {}).get(2)))

    # If ANY sub-chunk had code, flag the final chunk
    contains_code = any(c.get("metadata", {}).get("is_code_block", False) for c in current_buffer)

    merged_chunks.append({
        "content": merged_content,
        "word_count": total_words,
        "metadata": {
            # Keep the primary starting heading for standard reference
            "headings": current_buffer[0].get("metadata", {}).get("headings", {}),
            # Add a new field showing all sections this chunk spans across
            "sections_covered": {
                "h1": all_h1s,
                "h2": all_h2s
            },
            "is_image": False,
            "is_table": False,
            "contains_code_block": contains_code,
        }
    })

    # --- Handle the Overlap ---
    if overlap_chunks > 0 and len(current_buffer) > overlap_chunks:
        # Keep the last N chunks in the buffer for the next round
        # If your incoming chunks are ~60 words, overlap_chunks=1 leaves ~60 words in the buffer
        current_buffer[:] = current_buffer[-overlap_chunks:]
    else:
        current_buffer.clear()

def merge_rag_chunks(chunks: list, target_words: int = 300, max_words: int = 500, min_words: int = 150, max_table_rows: int = 15, overlap_chunks: int = 1) -> list:
    merged_chunks = []
    current_buffer = [] 
    
    # Helper to check top-level headings
    def get_top_headings(chunk):
        headings = chunk.get("metadata", {}).get("headings", {})
        return (headings.get(1), headings.get(2))

    for chunk in chunks:
        content = chunk.get("content", "").strip()
        if not content:
            continue

        chunk["word_count"] = count_words(content)
        meta = chunk.get("metadata", {})

        # --- Handle Atomic Chunks (Images and Tables) ---
        if meta.get("is_image", False):
            flush_text_buffer(merged_chunks, current_buffer, overlap_chunks=0)
            merged_chunks.extend(split_image_chunk(chunk)) # Assuming this function exists
            continue
            
        if meta.get("is_table", False):
            flush_text_buffer(merged_chunks, current_buffer, overlap_chunks=0)
            merged_chunks.extend(split_table_chunk(chunk, max_table_rows)) # Assuming this exists
            continue

        # --- Check for Semantic Boundaries with a MINIMUM SIZE limit ---
        if current_buffer:
            prev_headings = get_top_headings(current_buffer[-1])
            curr_headings = get_top_headings(chunk)
            current_words = sum(c["word_count"] for c in current_buffer)
            
            # ONLY flush on a topic change if the chunk is already big enough.
            # If it's too small, we swallow the boundary and keep building toward 300 words.
            if prev_headings != curr_headings and current_words >= min_words:
                flush_text_buffer(merged_chunks, current_buffer, overlap_chunks=0)

        # --- Handle Standard Text Chunks ---
        current_words = sum(c["word_count"] for c in current_buffer)
        
        # Flush if adding this chunk forces us over the absolute max_words limit
        if current_words > 0 and (current_words + chunk["word_count"] > max_words):
            flush_text_buffer(merged_chunks, current_buffer, overlap_chunks)

        current_buffer.append(chunk)
        
        # Flush if we hit our target size (e.g., 300 words)
        current_words = sum(c["word_count"] for c in current_buffer)
        if current_words >= target_words:
            flush_text_buffer(merged_chunks, current_buffer, overlap_chunks)

    # Flush any remaining text at the very end
    flush_text_buffer(merged_chunks, current_buffer, overlap_chunks=0)

    return merged_chunks

In [16]:
chunks = merge_rag_chunks(store)

In [ ]:
print(chunks)

In [18]:


def merge_rag_chunks(chunks: list, target_words: int = 300, max_words: int = 500) -> list:
    merged_chunks = []
    
    current_buffer = []
    current_word_count = 0
    current_metadata = {
        "headings": {},
        "is_image": False,
        "is_code_block": False,
        "is_table": False,
        "image_urls": [],
        "alt_texts": []
    }

    for chunk in chunks:
        content = chunk.get("content", "").strip()
        if not content:
            continue

        chunk_words = count_words(content)
        meta = chunk.get("metadata", {})

        # If adding this chunk exceeds max_words and buffer has content, flush current buffer first
        if current_word_count > 0 and (current_word_count + chunk_words > max_words):
            merged_chunks.append({
                "metadata": current_metadata.copy(),
                "content": "\n\n".join(current_buffer).strip(),
                "word_count": current_word_count
            })
            
            # Reset buffer with overlap (keep current chunk's heading context)
            current_buffer = []
            current_word_count = 0
            current_metadata = {
                "headings": meta.get("headings", {}).copy(),
                "is_image": False,
                "is_code_block": False,
                "is_table": False,
                "image_urls": [],
                "alt_texts": []
            }

        # Accumulate content and update metadata flags
        current_buffer.append(content)
        current_word_count += chunk_words
        
        current_metadata["headings"].update(meta.get("headings", {}))
        current_metadata["is_image"] = current_metadata["is_image"] or meta.get("is_image", False)
        current_metadata["is_code_block"] = current_metadata["is_code_block"] or meta.get("is_code_block", False)
        current_metadata["is_table"] = current_metadata["is_table"] or meta.get("is_table", False)
        
        if "image_urls" in meta:
            current_metadata["image_urls"].extend(meta["image_urls"])
        if "alt_texts" in meta:
            current_metadata["alt_texts"].extend(meta["alt_texts"])

        # Flush if we have hit or passed the target word count
        if current_word_count >= target_words:
            merged_chunks.append({
                "metadata": current_metadata.copy(),
                "content": "\n\n".join(current_buffer).strip(),
                "word_count": current_word_count
            })
            
            # Reset buffer
            current_buffer = []
            current_word_count = 0
            current_metadata = {
                "headings": meta.get("headings", {}).copy(),
                "is_image": False,
                "is_code_block": False,
                "is_table": False,
                "image_urls": [],
                "alt_texts": []
            }

    # Flush remaining leftover chunks
    if current_buffer:
        merged_chunks.append({
            "metadata": current_metadata.copy(),
            "content": "\n\n".join(current_buffer).strip(),
            "word_count": current_word_count
        })

    return merged_chunks

In [19]:
chunks =  merge_rag_chunks(store)

In [ ]:
print(chunks)

In [21]:
import re
from typing import List, Dict, Any

class MarkdownRAGSplitter:
    def __init__(
        self, 
        target_words: int = 250, 
        max_words: int = 400, 
        max_table_rows: int = 15
    ):
        self.target_words = target_words
        self.max_words = max_words
        self.max_table_rows = max_table_rows

    def _word_count(self, text: str) -> int:
        return len(re.findall(r'\w+', text))

    def _get_section_key(self, headings: Dict[int, str]) -> str:
        """Constructs a hard boundary key using H1 and H2 headings."""
        h1 = headings.get(1, "").strip()
        h2 = headings.get(2, "").strip()
        return f"{h1} > {h2}"

    def split_ast_chunks(self, ast_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        final_chunks = []
        
        current_buffer = []
        current_word_count = 0
        current_section_key = None
        current_metadata = {}

        for chunk in ast_chunks:
            content = chunk.get("content", "").strip()
            meta = chunk.get("metadata", {})
            if not content:
                continue

            headings = meta.get("headings", {})
            section_key = self._get_section_key(headings)
            chunk_words = self._word_count(content)

            # 1. SPECIAL CASE: Oversized Pipe Tables (> max_table_rows)
            if meta.get("is_table") and content.count("\n") > self.max_table_rows:
                # Flush existing buffer before handling the table
                if current_buffer:
                    final_chunks.append(self._build_chunk(current_buffer, current_metadata, current_word_count))
                    current_buffer, current_word_count = [], 0

                final_chunks.extend(self._slice_large_table(content, meta))
                current_section_key = section_key
                continue

            # 2. HARD BOUNDARY: H1/H2 Section Shift
            section_changed = (current_section_key is not None) and (section_key != current_section_key)

            # 3. CAPACITY OVERFLOW: Exceeds max_words
            overflows = (current_word_count + chunk_words) > self.max_words

            if (section_changed or overflows) and current_buffer:
                final_chunks.append(self._build_chunk(current_buffer, current_metadata, current_word_count))
                current_buffer = []
                current_word_count = 0

            # 4. OVERSIZED SINGLE ELEMENT (Paragraph/Code/Vision Text > max_words)
            if chunk_words > self.max_words and not meta.get("is_table"):
                sub_texts = self._split_text_element(content, self.max_words)
                for i, sub_text in enumerate(sub_texts):
                    sub_meta = meta.copy()
                    sub_meta["split_part"] = i + 1
                    final_chunks.append({
                        "metadata": sub_meta,
                        "content": sub_text,
                        "word_count": self._word_count(sub_text)
                    })
                current_section_key = section_key
                continue

            # 5. ACCUMULATE INTO BUFFER
            current_buffer.append(content)
            current_word_count += chunk_words
            current_section_key = section_key
            current_metadata = self._merge_metadata(current_metadata, meta, is_new=(len(current_buffer) == 1))

            # 6. FLUSH IF TARGET WORDS REACHED
            if current_word_count >= self.target_words:
                final_chunks.append(self._build_chunk(current_buffer, current_metadata, current_word_count))
                current_buffer = []
                current_word_count = 0

        # Flush any trailing buffer
        if current_buffer:
            final_chunks.append(self._build_chunk(current_buffer, current_metadata, current_word_count))

        return final_chunks

    def _slice_large_table(self, table_str: str, meta: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Splits a markdown table by rows while keeping the header intact."""
        lines = [line for line in table_str.strip().split("\n") if line.strip()]
        if len(lines) < 3:
            return [{"metadata": meta, "content": table_str, "word_count": self._word_count(table_str)}]

        header, delimiter, rows = lines[0], lines[1], lines[2:]
        sliced_chunks = []

        for i in range(0, len(rows), self.max_table_rows):
            batch = rows[i:i + self.max_table_rows]
            sliced_table = "\n".join([header, delimiter] + batch)
            
            sub_meta = meta.copy()
            sub_meta["table_slice"] = f"Rows {i + 1}-{i + len(batch)}"
            
            sliced_chunks.append({
                "metadata": sub_meta,
                "content": sliced_table,
                "word_count": self._word_count(sliced_table)
            })

        return sliced_chunks

    def _split_text_element(self, text: str, limit: int) -> List[str]:
        """Splits oversized paragraphs or code blocks along natural line breaks."""
        lines = text.split("\n")
        chunks, current, count = [], [], 0

        for line in lines:
            line_words = self._word_count(line)
            if count + line_words > limit and current:
                chunks.append("\n".join(current))
                current, count = [], 0
            current.append(line)
            count += line_words

        if current:
            chunks.append("\n".join(current))
        return chunks

    def _merge_metadata(self, existing: Dict, new_meta: Dict, is_new: bool) -> Dict:
        """Combines flags, URLs, and heading maps across merged AST nodes."""
        if is_new:
            res = new_meta.copy()
            res["image_urls"] = list(new_meta.get("image_urls", []))
            res["alt_texts"] = list(new_meta.get("alt_texts", []))
            return res

        merged = existing.copy()
        merged["headings"].update(new_meta.get("headings", {}))
        merged["is_image"] = merged.get("is_image", False) or new_meta.get("is_image", False)
        merged["is_code_block"] = merged.get("is_code_block", False) or new_meta.get("is_code_block", False)
        merged["is_table"] = merged.get("is_table", False) or new_meta.get("is_table", False)

        for url in new_meta.get("image_urls", []):
            if url not in merged["image_urls"]:
                merged["image_urls"].append(url)
        for alt in new_meta.get("alt_texts", []):
            if alt not in merged["alt_texts"]:
                merged["alt_texts"].append(alt)

        return merged

    def _build_chunk(self, buffer: List[str], meta: Dict, word_count: int) -> Dict[str, Any]:
        return {
            "metadata": meta,
            "content": "\n\n".join(buffer).strip(),
            "word_count": word_count
        }

In [22]:
splitter = MarkdownRAGSplitter()
splited = splitter.split_ast_chunks(store)

In [ ]:
print(splited)